# Retail Sales Analytics and Forecasting

This notebook uses the UCI Online Retail dataset (DOI: 10.24432/C5BW33), downloaded by the project pipeline. It covers validated cleaning, exploratory analysis, regression comparison, and a 30-day daily-sales forecast. See the project README for the source, license, and reproducibility instructions.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from retail_sales.analytics import daily_sales
from retail_sales.pipeline import run_pipeline

sns.set_theme(style="whitegrid")
results = run_pipeline(ROOT)
data = pd.read_csv(ROOT / "data/processed/retail_sales_clean.csv", parse_dates=["InvoiceDate"])
daily = daily_sales(data)
metrics = pd.read_csv(ROOT / "data/processed/model_metrics.csv")
forecast = pd.read_csv(ROOT / "data/processed/sales_forecast.csv", parse_dates=["date"]).set_index("date")
print(f"Loaded {results['raw_rows']:,} source rows; retained {results['clean_rows']:,} valid sales rows.")

/tmp/ipykernel_5789/2766860749.py:19: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(ROOT / "data/processed/retail_sales_clean.csv", parse_dates=["InvoiceDate"])


Loaded 541,909 source rows; retained 524,878 valid sales rows.


## Data quality and validation

Exact duplicates, cancellations, and rows with non-positive quantity or price are excluded from positive-sales analysis. High sales outliers are flagged using the 1.5-IQR rule and retained.

In [2]:
display(pd.Series(results["cleaning_report"], name="count").to_frame())
display(data[["InvoiceDate", "Quantity", "UnitPrice", "Sales"]].describe().T)
print("Missing values in cleaned data:", int(data.isna().sum().sum()))
print("Duplicate rows in cleaned data:", int(data.duplicated().sum()))

,count
input_rows,541909
rows_missing_required_identifiers,0
duplicate_rows_removed,5268
rows_removed_as_invalid_or_returns,11763
valid_sales_rows,524878
sales_outliers_flagged,42624


,count,mean,min,25%,50%,75%,max,std
InvoiceDate,524878,2011-07-04 15:30:16.317049,2010-12-01 08:26:00,2011-03-28 12:13:00,2011-07-20 11:22:00,2011-10-19 11:41:00,2011-12-09 12:50:00,NaN
Quantity,524878.0,10.6166,1.0,1.0,4.0,11.0,80995.0,156.280031
UnitPrice,524878.0,3.922573,0.001,1.25,2.08,4.13,13541.33,36.093028
Sales,524878.0,20.275399,0.001,3.9,9.92,17.7,168469.6,271.693566


Missing values in cleaned data: 132186


Duplicate rows in cleaned data: 0


## Exploratory analysis

Revenue is line sales in GBP. Country is the available regional field in the source.

In [3]:
monthly_sales = daily.resample("MS").sum()
weekday_sales = data.assign(Weekday=data["InvoiceDate"].dt.day_name()).groupby("Weekday")["Sales"].sum()
weekday_sales = weekday_sales.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly_sales.plot(ax=axes[0], marker="o", color="#126e75")
axes[0].set(title="Monthly Sales", ylabel="Sales (GBP)", xlabel="Month")
weekday_sales.plot.bar(ax=axes[1], color="#d47a45")
axes[1].set(title="Sales by Weekday", ylabel="Sales (GBP)", xlabel="")
plt.tight_layout()

In [4]:
top_products = data.groupby("Description")["Sales"].sum().nlargest(10).sort_values()
top_countries = data.groupby("Country")["Sales"].sum().nlargest(10).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top_products.plot.barh(ax=axes[0], color="#d47a45")
axes[0].set(title="Top Products by Sales", xlabel="Sales (GBP)")
top_countries.plot.barh(ax=axes[1], color="#548c69")
axes[1].set(title="Top Countries by Sales", xlabel="Sales (GBP)")
plt.tight_layout()

correlation = data[["Quantity", "UnitPrice", "Sales"]].corr()
sns.heatmap(correlation, annot=True, cmap="vlag", center=0, fmt=".2f")
plt.title("Transaction Metric Correlations")
plt.tight_layout()

## Model comparison and forecast

Both regressors are evaluated on the final chronological 20% of complete daily observations. The lowest-MAE estimator is refit on all observations and recursively forecast for 30 days.

In [5]:
display(metrics.sort_values("mae").reset_index(drop=True))
print("Selected model:", results["best_model"])

fig, axis = plt.subplots(figsize=(12, 5))
daily.tail(90).plot(ax=axis, label="Actual", color="#126e75")
forecast["sales"].plot(ax=axis, label="Forecast", color="#d47a45")
axis.set(title="Daily Sales and 30-Day Forecast", xlabel="Date", ylabel="Sales (GBP)")
axis.legend()
plt.tight_layout()
display(forecast.head())

,model,mae,rmse,r2
0,Random Forest,13387.847312,23573.842403,0.410796
1,Linear Regression,15218.571396,23906.559351,0.394047


Selected model: Random Forest


,sales
date,
2011-12-10,0.000000
2011-12-11,26106.305142
2011-12-12,70145.396541
2011-12-13,77486.168776
2011-12-14,73339.332485
